In [0]:
import os

_classroom_setup_common_nb_path = (
    dbutils.notebook.entry_point
    .getDbutils()
    .notebook()
    .getContext()
    .notebookPath()
    .get()
)

# Under %run this is the calling lesson, which sits at the imported course root. Import can rename
# that folder, so data paths must not repeat the course folder name.
_COURSE_ROOT_DIR = "/Workspace" + os.path.dirname(_classroom_setup_common_nb_path)

In [0]:
%sql
-- Create safe function for catalog name
CREATE OR REPLACE TEMPORARY FUNCTION safe_uc_name(value STRING)
RETURNS STRING
RETURN
  COALESCE(
    NULLIF(
      REGEXP_REPLACE(
        REGEXP_REPLACE(
          LOWER(TRIM(value)),
          '[^a-z0-9_]',
          '_'
        ),
        '_+',
        '_'
      ),
      ''
    ),
    'user'
  );

-- Create temp view of all available catalogs
CREATE OR REPLACE TEMPORARY VIEW list_of_catalogs AS
SELECT LOWER(catalog_name) AS catalog_names
FROM system.information_schema.catalogs;

In [0]:
%sql
BEGIN

-- =========================================
-- 1. Declare variables
-- =========================================
  DECLARE user_email STRING;
  DECLARE user_name STRING;
  DECLARE safe_user_name STRING;
  DECLARE vocareum_catalog_name STRING DEFAULT 'NA';
  DECLARE catalog_name STRING DEFAULT 'NA';
  DECLARE catalog_already_exists BOOLEAN DEFAULT FALSE;
  DECLARE create_catalog_statement STRING DEFAULT '';
  DECLARE create_view_with_catalog_name STRING DEFAULT '';
  DECLARE test STRING DEFAULT '';

-- =========================================
-- 2. Capture current user
-- =========================================
  SET user_email = current_user();
  SET user_name = SPLIT(user_email, '@')[0];
  SET safe_user_name = (SELECT safe_uc_name(SPLIT(current_user(), '@')[0]));

-- ==================================================================================
-- 3. Determine Vocareum or Non-Vocareum workspace and set/create catalog
-- ==================================================================================
  IF (LOWER(user_email) LIKE '%@vocareum.com')
    THEN
      SET vocareum_catalog_name = safe_user_name;

      IF EXISTS (
        SELECT 1
        FROM list_of_catalogs
        WHERE catalog_names = LOWER(vocareum_catalog_name)
      ) THEN
        SET catalog_already_exists = TRUE;
        SET create_view_with_catalog_name = CONCAT(
              "CREATE OR REPLACE TEMP VIEW catalog_name_vw AS ",
              "SELECT '", vocareum_catalog_name, "' AS catalog_name"
            );
      END IF;

  ELSE
    SET catalog_name = CONCAT('learn_get_started_da_', safe_user_name);
    SET create_view_with_catalog_name = CONCAT(
      "CREATE OR REPLACE TEMP VIEW catalog_name_vw AS ",
      "SELECT '", catalog_name, "' AS catalog_name"
    );

    IF EXISTS (
      SELECT 1
      FROM list_of_catalogs
      WHERE catalog_names = LOWER(catalog_name)
    )
      THEN
        SET catalog_already_exists = TRUE;
        SET test = 'catalog exists';
    ELSE
      SET create_catalog_statement = CONCAT(
        'CREATE CATALOG IF NOT EXISTS ',
        catalog_name
      );
      SET test = 'catalog does not exist';
      EXECUTE IMMEDIATE create_catalog_statement;
    END IF;

  END IF;

  EXECUTE IMMEDIATE create_view_with_catalog_name;

-- ===================================================================
-- 4. Return final results
-- ===================================================================
  SELECT
    CASE
      WHEN vocareum_catalog_name <> 'NA' THEN 'Vocareum Workspace'
      ELSE 'Learner Workspace'
    END AS Workspace,
    CASE
      WHEN vocareum_catalog_name <> 'NA' THEN vocareum_catalog_name
      ELSE catalog_name
    END AS `User Catalog Name`,
    CASE
      WHEN catalog_already_exists = TRUE THEN 'Catalog Already Exists'
      ELSE 'Catalog Created'
    END AS `Catalog Status`;
END;

In [0]:
%sql
-- Declare and set the catalog variable from the temp view
DECLARE OR REPLACE my_catalog STRING;
SET VAR my_catalog = (SELECT catalog_name FROM catalog_name_vw);

-- Set the active catalog
USE CATALOG IDENTIFIER(my_catalog);

In [0]:
my_catalog = spark.sql("SELECT current_catalog()").first()[0]

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS retail;
USE SCHEMA retail;

In [0]:
SCHEMA_NAME_RETAIL = "retail"
VOLUME_NAME_RETAIL = "data"

## Create volume inside the schema
spark.sql(f"CREATE VOLUME IF NOT EXISTS `{my_catalog}`.`{SCHEMA_NAME_RETAIL}`.`{VOLUME_NAME_RETAIL}`")